# Normative modelling tutorial: solutions

Solutions to the exercises in `normative_modelling_tutorial.ipynb`.

Run the first code cell before any solution. It runs the whole tutorial notebook, so that all its data, models and z-scores exist here too. This takes about half a minute. The tutorial's own plots are switched off while it runs.

In [ ]:
# Run all code cells of the tutorial notebook, so that its data, models and
# z-scores exist here too. Its plots and printed output are switched off while it runs.
import io
import json
from contextlib import redirect_stdout
import matplotlib.pyplot as plt

plt.switch_backend("agg")                     # plots off
tutorial = json.load(open("normative_modelling_tutorial.ipynb"))
with redirect_stdout(io.StringIO()):          # printed output off
    for cell in tutorial["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]))
plt.close("all")
%matplotlib inline
print("Tutorial notebook loaded.")

### 2a. The "average patient"

In [ ]:
# Solution 2a — the group effect is diluted by the unaffected patients
roi = "entorhinal"
j = ROIS.index(roi)
control_mean = adult_ctrl[roi].mean()
control_sd = adult_ctrl[roi].std()

abnormal = patients[truth[:, j]]       # patients truly abnormal in this region
normal = patients[~truth[:, j]]        # all other patients

d_all = (patients[roi].mean() - control_mean) / control_sd
d_abnormal = (abnormal[roi].mean() - control_mean) / control_sd
d_normal = (normal[roi].mean() - control_mean) / control_sd

print(f"all patients      (n={len(patients)}): d = {d_all:.2f}")
print(f"truly abnormal    (n={len(abnormal)}): d = {d_abnormal:.2f}")
print(f"not abnormal here (n={len(normal)}): d = {d_normal:.2f}")
# The "all patients" d is roughly a weighted average of the two groups: a large effect in
# a minority of patients looks like a modest effect in "the average patient".
# (d here is not adjusted for age or site, so the numbers are only approximate.)

### 3a. Flexibility vs. overfitting

In [ ]:
# Solution 3a — flexibility vs. overfitting
# Refit the model with more and more knots and track test-set MSLL.
roi = "precuneus"
knots = [3, 4, 6, 10, 15, 30]
msll = []
for k in knots:
    model = fit_normative_model(train, train[roi].values, n_knots=k)
    mu, sd = predict_normative_model(model, test)
    msll.append(evaluate(test[roi].values, mu, sd, train[roi].values)["MSLL"])

plt.figure(figsize=(5, 3))
plt.plot(knots, msll, "o-")
plt.xlabel("n_knots")
plt.ylabel("test MSLL (lower = better)")
plt.title("3a")
plt.show()
# MSLL improves up to about 6 knots and then slowly gets worse: the extra flexibility
# starts fitting noise in the training data, which doesn't carry over to held-out people.

### 3b. Ignore the site

In [ ]:
# Solution 3b — ignoring site
roi = "precuneus"
y_tr = train[roi].values
y_te = test[roi].values

model_site = fit_normative_model(train, y_tr)
mu, sd = predict_normative_model(model_site, test)
z_site = (y_te - mu) / sd

model_nosite = fit_normative_model(train, y_tr, site_in_mean=False, site_in_var=False)
mu, sd = predict_normative_model(model_nosite, test)
z_nosite = (y_te - mu) / sd

# % of |z| > 1.96 per site: should be ~5 everywhere.
# Without site terms, sites with a larger/smaller spread or a shifted mean get too many/too few
# "extreme" z-scores. If patients come mostly from one site, site effects masquerade as disease effects.
pd.DataFrame({"with site": 100 * pd.Series(np.abs(z_site) > 1.96).groupby(test["site"]).mean(),
              "no site": 100 * pd.Series(np.abs(z_nosite) > 1.96).groupby(test["site"]).mean()}).round(1)

### 3c. Variance model

In [ ]:
# Solution 3c — site effect on the mean only, not on the variance
roi = "precuneus"
model = fit_normative_model(train, train[roi].values, site_in_var=False)
mu, sd = predict_normative_model(model, test)
z = (test[roi].values - mu) / sd

# SD of the z-scores per site should be ~1. Without a site term in the variance model
# it drifts above 1 at noisy sites and below 1 at quiet sites.
pd.Series(z).groupby(test["site"]).std().round(2)

### 3d. One person on the chart

In [ ]:
# Solution 3d — one person on the chart
roi = "entorhinal"
j = ROIS.index(roi)
model = fit_normative_model(train, train[roi].values)

one_patient = patients.iloc[[0]]            # double brackets keep it a (1-row) data frame
mu, sd = predict_normative_model(model, one_patient)
y = one_patient[roi].values
z = (y - mu) / sd
centile = 100 * stats.norm.cdf(z)

print(f"age {one_patient['age'].values[0]:.0f}, {one_patient['site'].values[0]}")
print(f"observed thickness: {y[0]:.2f} mm")
print(f"expected thickness: {mu[0]:.2f} mm (SD {sd[0]:.2f})")
print(f"z = {z[0]:.2f}  ->  centile {centile[0]:.1f}")
print("truly abnormal in this region:", truth[0, j])
# A z-score tells us how unusual this person is compared with healthy people of the same
# age, sex and site. The centile says the same thing on the familiar growth-chart scale.
# (Section 3's Gaussian model gets the tails of skewed regions like entorhinal slightly wrong.)

### 3e. Where does the chart come from?

In [ ]:
# Solution 3e — which parts of the chart are backed by data?
roi = "precuneus"
model = fit_normative_model(train, train[roi].values)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, site in zip(axes, ["site_A", "site_B", "site_C"]):
    plot_centiles(model, roi, ax, site=site)
plt.tight_layout()
plt.show()
# The grey dots show where each site has data: site_A 6-25, site_B 18-60, site_C 40-88.
# The curves are drawn for ALL ages, but outside a site's own age range they rest on the
# shared age curve plus that site's offset. That assumes the site effect is the same at
# every age, which we can't check without data. Treat those parts as extrapolation.

### 4a. No site effect on the variance

In [ ]:
# Solution 4a — warped BLR without a site/sex effect on the variance
novar_template = BLR(
    basis_function_mean=BsplineBasisFunction(degree=3, nknots=5),
    heteroskedastic=True,
    fixed_effect=True,
    fixed_effect_var=False,        # <- the only change from Model 2
    warp_name="WarpSinhArcsinh",
    warp_reparam=True,
)
novar = NormativeModel(novar_template, savemodel=False, saveresults=False, saveplots=False)
nd_train_novar = to_normdata("train", train)
nd_test_novar = to_normdata("test", test)
novar.fit_predict(nd_train_novar, nd_test_novar)
Z_novar = nd_test_novar.Z.values

# % |z| > 1.96 per site for one region, with vs. without site-specific variance
j = ROIS.index("precuneus")
# Sites with noisier scanners get too many "extreme" deviations, quieter sites too few.
pd.DataFrame({
    "site var (full model)": 100 * pd.Series(np.abs(Z_wblr_test[:, j]) > 1.96).groupby(test["site"]).mean(),
    "no site var": 100 * pd.Series(np.abs(Z_novar[:, j]) > 1.96).groupby(test["site"]).mean()}).round(1)

### 4b. Q–Q plots of the Gaussian BLR model

In [ ]:
# Solution 4b — Q-Q plots of the Gaussian BLR model
plot_qq(nd_test_blr, plot_id_line=True);
# The skewed regions (entorhinal, cingulate, lat_occipital, insula) bend away from the line
# in the tails, just like our own Gaussian model in 3.4. With the warp (4.4) they follow the line.
# These are the regions with the largest z_skew for "PCN BLR" in the table in 4.3.

### 5a. False positives in each tail

In [ ]:
# Solution 5a — tail-specific false-positive rates in healthy controls
# For a calibrated model each tail should contain ~0.47% of controls (z < -2.6 or z > +2.6).
fp_tab = pd.DataFrame(index=ROIS)
fp_tab["Gaussian: % z<-2.6"] = (100 * (Z_test < -2.6).mean(0)).round(2)
fp_tab["Gaussian: % z>+2.6"] = (100 * (Z_test > 2.6).mean(0)).round(2)
fp_tab["warped BLR: % z<-2.6"] = (100 * (Z_wblr_test < -2.6).mean(0)).round(2)
fp_tab["warped BLR: % z>+2.6"] = (100 * (Z_wblr_test > 2.6).mean(0)).round(2)
# With positive skew, the Gaussian model over-flags the upper tail and under-flags the lower tail,
# so it would miss genuine thinning in skewed regions.
fp_tab

### 5c. A deviation profile for one patient

In [ ]:
# Solution 5c — a deviation profile for one patient
i = order[0]                     # the patient with the most extreme deviations

plt.figure(figsize=(8, 3))
plt.bar(ROIS, Zp[i])
plt.axhline(-2.6, color="red", ls="--", label="z = -2.6")
plt.axhline(0, color="black", lw=0.8)
plt.xticks(rotation=45, ha="right")
plt.ylabel("z")
plt.title(f"Deviation profile of patient {i}")
plt.legend()
plt.show()

flagged = [ROIS[j] for j in range(N_ROI) if Zp[i, j] < -2.6]
truly_abnormal = [ROIS[j] for j in range(N_ROI) if truth[i, j]]
print("flagged (z < -2.6):", flagged)
print("truly abnormal:    ", truly_abnormal)
# For a patient with no extreme deviations, try e.g. i = order[-1].
# Explaining it: each bar says how far this person is from healthy people of the same age,
# sex and scanner, in SD units. Bars beyond the line are unusual (about 1 in 200 healthy people),
# not proof of disease.